# BATTWIN Probabilistic workflow
Use the frozen models first. See [the tutorial](../docs/TUTORIAL.md) for development and design protocols.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT/"src"))
from battwin_probabilistic import Project, SimulationConfig
from battwin_probabilistic import reporting
project = Project(ROOT)
project.catalog.registry().drop(columns="bundle_path")

In [ ]:
data = project.data("mixing")
model = project.model("mixing")
test = data["test"]
predictions = model.predict(test)
predictions.head()

In [ ]:
target = model.targets[0]
p = predictions[predictions.target.eq(target)]
reporting.parity(test[target].to_numpy(), p["median"].to_numpy(), "Viscosity (Pa.s)", log=True, path=ROOT/"runs/mixing/figures/parity");

In [ ]:
reporting.uncertainty(test[target].to_numpy(), p["median"].to_numpy(), p.q05.to_numpy(), p.q95.to_numpy(), "Viscosity (Pa.s)", log=True, path=ROOT/"runs/mixing/figures/uncertainty");

In [ ]:
from battwin_probabilistic.data import read_table
inputs = read_table(ROOT/"examples/data/chain_inputs.csv")
chain = project.chain(SimulationConfig(particles=40, seed=42))
summary, particles = chain.simulate(inputs)
summary.head()

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from battwin_probabilistic import TuningConfig
    artifact = project.train("mixing", ROOT/"runs/mixing/new_search", use_selected=False, tuning=TuningConfig(trials=50))

In [ ]:
RUN_DESIGN = False
if RUN_DESIGN:
    from battwin_probabilistic.design_space import SearchConfig
    from battwin_probabilistic.optimisation import optimise_capacity
    search = optimise_capacity(project.data("electrochemical")["development"], SearchConfig(discharge_rate=.2), project.catalog, ROOT/"runs/design/0.2C_capped")